## Demo the usage of GeoPandas with PySpark.

Package that might need installation:

```shell
pip install geopandas xyzservices folium matplotlib mapclassify
```

In [ ]:
from spark_esri import spark_start, spark_stop

In [ ]:
import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz

import geofunctions as S

In [ ]:
config = {
    "spark.driver.memory": "32G",
    "spark.executor.memory": "32G",
    "spark.jars": "geofunctions-0.6.jar",
}

spark_stop()
spark = spark_start(config=config)

### Convert random lon/lat to web mercator x/y.

In [ ]:
xmin, xmax = -180.0, 180.0
ymin, ymax = -80.0, 80.0
xdel = xmax - xmin
ydel = ymax - ymin

df = (
    spark.range(10_000)
    .withColumn("x", S.st_lontox(F.rand() * xdel + xmin))
    .withColumn("y", S.st_lattoy(F.rand() * ydel + ymin))
)

### Convert x/y to point geometry, and get pandas dataframe.

In [ ]:
pdf = df.withColumn("geometry", S.st_point("x", "y")).drop("x", "y").toPandas()

### Convert pandas dataframe to geopandas dataframe.    

In [ ]:
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="EPSG:3857")
# Enable the spatial index.
_ = gdf.sindex

### Plot and explore the points.

In [ ]:
gdf.explore(tiles=xyz.Esri.WorldGrayCanvas)